In [1]:

# 1. Импорт библиотек

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import VarianceThreshold



# 2. Загрузка данных

sheet_id = "1q-nbWuFrfrIBMXmZfNW78N3bx5v60Vb9"
url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv"

df = pd.read_csv(url)



# 3. Очистка данных

df = df.drop(columns=['Unnamed: 0'])
df = df.fillna(df.median(numeric_only=True))

selector = VarianceThreshold(threshold=0.01)
X_temp = df.drop(columns=['IC50, mM', 'CC50, mM', 'SI'])

selector.fit(X_temp)

low_var_cols = X_temp.columns[~selector.get_support()]
df = df.drop(columns=low_var_cols)



# 4. Подготовка таргета

cc50_median = df['CC50, mM'].median()
df['CC50_class'] = (df['CC50, mM'] > cc50_median).astype(int)

X = df.drop(columns=['IC50, mM', 'CC50, mM', 'SI', 'CC50_class'])
y = df['CC50_class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)



# 5. Logistic Regression

logreg = LogisticRegression(max_iter=1000)
logreg.fit(X_train, y_train)

y_pred_lr = logreg.predict(X_test)

print("Logistic Regression")
print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))



# 6. Random Forest

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

print("\nRandom Forest")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

/tmp/ipykernel_4736/1709003372.py:42: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['CC50_class'] = (df['CC50, mM'] > cc50_median).astype(int)
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", 

Logistic Regression
Accuracy: 0.47761194029850745
              precision    recall  f1-score   support

           0       0.48      1.00      0.65        96
           1       0.00      0.00      0.00       105

    accuracy                           0.48       201
   macro avg       0.24      0.50      0.32       201
weighted avg       0.23      0.48      0.31       201


Random Forest
Accuracy: 0.8059701492537313
              precision    recall  f1-score   support

           0       0.77      0.85      0.81        96
           1       0.85      0.76      0.80       105

    accuracy                           0.81       201
   macro avg       0.81      0.81      0.81       201
weighted avg       0.81      0.81      0.81       201



### Что получилось

**Logistic Regression**
- Accuracy: **0.478**
- Модель предсказывает почти только класс **0**
- Класс 1 не определяется:
   precision = 0.00
   recall = 0.00

**Random Forest**
- Accuracy: **0.806**
- Класс 0:
   precision: 0.77
   recall: 0.85
- Класс 1:
   precision: 0.85
   recall: 0.76

### Вывод

- Logistic Regression снова работает плохо и уходит в один класс  
- Random Forest даёт хороший и сбалансированный результат  
- Классификация **CC50 > медианы** решается лучше, чем регрессия CC50  

 для этой задачи оставляю **Random Forest** как лучшую модель

### Финальный вывод по классификации CC50

- Лучшая модель: **Random Forest**
- Accuracy: **≈ 0.81**
- Модель даёт сбалансированные результаты по обоим классам  

- Logistic Regression не справилась (предсказывает один класс)  

 классификация работает лучше, чем регрессия CC50  

Следующее:
- создаю файл **classification_si.ipynb**